# Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from category_encoders import OneHotEncoder, GLMMEncoder, TargetEncoder, CatBoostEncoder
from sklearn import set_config
from colorama import Style, Fore
import math
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, cross_val_predict
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.metrics import roc_auc_score, roc_curve, make_scorer, f1_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer, KNNImputer
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.preprocessing import FunctionTransformer, StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from tabulate import tabulate
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.gaussian_process import GaussianProcessClassifier
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

sns.set_theme(style = 'white', palette = 'tab10')
pal = sns.color_palette('viridis')

pd.set_option('display.max_rows', 100)
set_config(transform_output = 'pandas')
pd.options.mode.chained_assignment = None
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


# Data

In [ ]:
train = pd.read_csv(r'/kaggle/input/ml-olympiad-predicting-earthquake-damage/train.csv',index_col='building_id')
test = pd.read_csv(r'/kaggle/input/ml-olympiad-predicting-earthquake-damage/test.csv',index_col='building_id')

In [ ]:
print(f'{Style.BRIGHT}{Fore.YELLOW} SHAPE')
print(f'{Style.BRIGHT}{Fore.YELLOW} -> Train: {Fore.GREEN} {train.shape}')
print(f'{Style.BRIGHT}{Fore.YELLOW} -> Test:  {Fore.GREEN} {test.shape}')

print(f'\n\n{Style.BRIGHT}{Fore.YELLOW} NULL VALUES')
print(f'{Style.BRIGHT}{Fore.YELLOW} -> Train: {Fore.GREEN} {train.isnull().any().any()}')
print(f'{Style.BRIGHT}{Fore.YELLOW} -> Test:  {Fore.GREEN} {test.isnull().any().any()}')

print(f'\n\n{Style.BRIGHT}{Fore.YELLOW} DUPLICATES')
print(f'{Style.BRIGHT}{Fore.YELLOW} -> Train: {Fore.GREEN} {train.duplicated().any().any()}')
print(f'{Style.BRIGHT}{Fore.YELLOW} -> Test:  {Fore.GREEN} {test.duplicated().any().any()}')

In [ ]:
train.head(3)

In [ ]:
test.head(3)

# Descriptive statistics

In [ ]:
def custom_describe(data):    
    desc = pd.DataFrame(index = list(data))
    #desc['type'] = data.dtypes
    #desc['count'] = data.count()
    #desc['nunique'] = data.nunique()
    #desc['%unique'] = desc['nunique'] /len(train) * 100
    #desc['null'] = train.isnull().sum()
    #desc['%null'] = desc['null'] / len(data) * 100
    desc = pd.concat([desc,data.describe().T.drop('count',axis=1)],axis=1)
    
    print(tabulate(desc, headers=desc.columns.tolist(),tablefmt='simple'))

In [ ]:
(custom_describe(train))

In [ ]:
SEED = 42
TARGET = 'damage_grade'
NUMERIC_COLS = [f for f in train._get_numeric_data() if f != TARGET]
CAT_COLS = list(test.drop(NUMERIC_COLS,axis=1))

# Correlation

In [ ]:
def plot_correlation(method='spearman'):
    corr = train[NUMERIC_COLS+[TARGET]].corr(method=method)
    plt.figure(figsize = (15, 15), dpi = 300)
    mask = np.zeros_like(corr)
    mask[np.triu_indices_from(mask)]=True
    sns.heatmap(data=corr, mask=mask, annot=True, cmap='YlOrBr',annot_kws = {'size' : 6})

In [ ]:
plot_correlation()

# Hierarchial Clustering

In [ ]:
def distance(data, label = '',method='spearman'):
    corr = data.corr(method = method)
    dist_linkage = linkage(squareform(1 - abs(corr)), 'complete')
    
    plt.figure(figsize = (10, 8))
    dendro = dendrogram(dist_linkage, labels=data.columns, leaf_rotation=90)
    plt.title(f'Feature Distance in {label} Dataset', weight = 'bold', size = 15)
    plt.show()

In [ ]:
distance(train[NUMERIC_COLS+[TARGET]],'train')

In [ ]:
distance(test[NUMERIC_COLS],'test')

# EDA

In [ ]:
def plot_numeric():
    n_cols = 3
    n_rows = math.ceil(len(NUMERIC_COLS)/n_cols)
    fig, ax = plt.subplots(n_rows,n_cols, figsize=(20,n_rows*3.5))
    ax = ax.flatten()
    plt.subplots_adjust(wspace=0.2,hspace=0.5)
    for i,c in enumerate(NUMERIC_COLS):
        ax[i].hist(train[c],color='red',alpha=0.5,density=True)
        ax[i].hist(test[c],color='green',alpha=0.5,density=True)
        ax[i].set_title(f'{c}')
    for j in range(len(NUMERIC_COLS),len(ax)):
        ax[j].axis('off')
    fig.suptitle('Distribution of Feature\nper Dataset\n', fontsize = 14 ,fontweight = 'bold',y=0.90)
    fig.legend(['Train', 'Test'],loc='right', bbox_to_anchor=[0.95,0.90])
    plt.show()


In [ ]:
plot_numeric()

* We saw in adversarial validation that the training and test data possibly do not follow the same distribution, however, visually some numerical variables have the same distribution in both.

# Categorical Features


In [ ]:
def plot_categorical():
    fig, ax = plt.subplots(len(CAT_COLS), 2, figsize = (14, len(CAT_COLS)*5))

    for i, column in enumerate(CAT_COLS):
        ax[i][0].pie(
            train[column].value_counts(), 
            shadow = True, 
            explode = [.1 for i in range(train[column].nunique())], 
            autopct = '%1.f%%',
            textprops = {'size' : 14, 'color' : 'white'}
        )
        
        sns.countplot(x=train[column], ax=ax[i][1])
        ax[i][1].set_title(f'{column}')
        ax[i][1].set_ylabel('Count')
        ax[i][1].set_xlabel(column)
        ax[i][1].tick_params(axis='x', labelrotation=45)

        
    plt.tight_layout()
    plt.show()

In [ ]:
plot_categorical()

# Target

In [ ]:
ax = train[TARGET].value_counts().sort_values().plot(kind='bar')
ax.bar_label(ax.containers[0], label_type = 'center',color='white');
plt.ylabel('Freq');

* As we can see, this is a dataset with unbalanced classes, in this case we will choose to use StratifiedKfold in our validations.

# Cross Validate

In [ ]:
le=LabelEncoder()
le.fit_transform(train[TARGET])
train[TARGET] = le.transform(train[TARGET])

In [ ]:
def cross_validate(estimator, label = ''):    
    cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
    X = train.copy()
    y = X.pop(TARGET)
        
    val_predictions = np.zeros((len(X)))
    train_scores, val_scores = [], []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        
        model = clone(estimator)
    
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]
        
        X_val = X.iloc[val_idx]
        y_val = y.iloc[val_idx]       

        model.fit(X_train, y_train)
        
        train_preds = model.predict(X_train)
        val_preds = model.predict(X_val)
        if label=='cb':
            val_preds =val_preds.ravel()
        
        val_predictions[val_idx] += val_preds        
        
        train_score = f1_score(y_train, train_preds,average='macro')
        val_score = f1_score(y_val, val_preds,average='macro')
        
        train_scores.append(train_score)
        val_scores.append(val_score)
    
    print(f'Val Score: {np.mean(val_scores):.5f} ± {np.std(val_scores):.5f} | Train Score: {np.mean(train_scores):.5f} ± {np.std(train_scores):.5f} | {label}')
    
    return val_scores, val_predictions

In [ ]:
score, oof = pd.DataFrame(), pd.DataFrame()

# Models

In [ ]:
models = [
    ('log', LogisticRegression(random_state = SEED, max_iter = 1000000)),
    ('bnb', BernoulliNB()),
    ('rf', RandomForestClassifier(random_state = SEED)),
    ('et', ExtraTreesClassifier(random_state = SEED)),
    ('xgb', XGBClassifier(random_state = SEED)),
    ('lgb', LGBMClassifier(random_state = SEED,verbosity=0)),
    ('gb', GradientBoostingClassifier(random_state = SEED)),
    ('hgb', HistGradientBoostingClassifier(random_state = SEED))
]

for (label, model) in models:
    score[label], oof[label] =  cross_validate(
        make_pipeline(TargetEncoder(cols = CAT_COLS), SimpleImputer(), model),
        label = label)
    

In [ ]:
voting = VotingClassifier(estimators=models,voting='soft')
score['mean_models'], oof['mean_models'] =  cross_validate(
    make_pipeline(TargetEncoder(cols = CAT_COLS), SimpleImputer(), voting),
    label = 'mean_models')

# Score Models

In [ ]:
ax = score.mean().sort_values(ascending=True).plot(kind='barh')
ax.bar_label(ax.containers[0],label_type ='center',color='white',fontweight='bold');
ax.patches[-1].set_facecolor('green');
ax.set_title('Score models',fontweight='bold');
ax.set_xlabel('F1 macro (high is better)',fontweight='bold');


# Shap

In [ ]:
X = train.copy()
y = X.pop(TARGET)
preprocess = Pipeline([('cat',TargetEncoder(cols = CAT_COLS)),('imputer',SimpleImputer())])
X = preprocess.fit_transform(X,y)
model = RandomForestClassifier(random_state = SEED)
model.fit(X,y)


In [ ]:
explainer = shap.TreeExplainer(model)
X_test = preprocess.transform(test)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)

In [ ]:
shap.initjs()
shap.summary_plot(shap_values[0],X_test.values,feature_names=X_test.columns)